In [ ]:
import csv
import logging
import os
from typing import List, Dict
from yt_dlp import YoutubeDL

INPUT_FILE = "ytb_link.txt"
OUTPUT_FILE = "ytb.csv"
MAX_COMMENTS = None  

YDL_OPTS = {
    "getcomments": True,       
    "skip_download": True,     
    "quiet": True,             
    "no_warnings": True,
    "ignoreerrors": True,      
    "extract_flat": False,     
}

# Cấu hình Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def save_to_csv(data: List[Dict[str, str]], filepath: str):
    """Lưu danh sách dictionary vào CSV (Append mode)."""
    if not data:
        return

    file_exists = os.path.exists(filepath)
    mode = "a" if file_exists else "w"

    with open(filepath, mode, newline="", encoding="utf-8-sig") as f:
        fieldnames = ["url", "cmt"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()
        
        writer.writerows(data)

def fetch_comments(ydl: YoutubeDL, url: str) -> List[Dict[str, str]]:
    """Xử lý 1 URL và trả về danh sách comment."""
    results = []
    try:
        info = ydl.extract_info(url, download=False)
        
        # Nếu không lấy được info hoặc không có comment
        if not info:
            return []
            
        comments_data = info.get("comments")
        if not comments_data:
            logger.warning(f"Video không có comment hoặc bị tắt: {url}")
            return []

        # Trích xuất dữ liệu
        count = 0
        for c in comments_data:
            text = c.get("text")
            if text:
                results.append({
                    "url": url,
                    "cmt": text
                })
                count += 1
                
            if MAX_COMMENTS and count >= MAX_COMMENTS:
                break
                
    except Exception as e:
        logger.error(f"Lỗi khi xử lý {url}: {e}")

    return results

def main():
    # 1. Kiểm tra file input
    if not os.path.exists(INPUT_FILE):
        logger.error(f"Không tìm thấy file input: {INPUT_FILE}")
        return

    # 2. Đọc danh sách URL
    with open(INPUT_FILE, "r") as f:
        urls = [line.strip() for line in f if line.strip()]

    logger.info(f"Đã tìm thấy {len(urls)} video cần xử lý.")

    # 3. Khởi tạo YoutubeDL (Context Manager)
    with YoutubeDL(YDL_OPTS) as ydl:
        for idx, url in enumerate(urls):
            logger.info(f"[{idx+1}/{len(urls)}] Đang tải: {url}")
            
            # Lấy dữ liệu
            comments = fetch_comments(ydl, url)
            
            # Lưu ngay sau khi lấy xong từng video
            if comments:
                save_to_csv(comments, OUTPUT_FILE)
                logger.info(f"   -> Đã lưu {len(comments)} comment.")
            else:
                logger.info("   -> Không có dữ liệu để lưu.")

    logger.info(f"Hoàn tất! File kết quả: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

09:20:50 [INFO] Đã tìm thấy 16 video cần xử lý.
09:20:50 [INFO] [1/16] Đang tải: https://www.youtube.com/watch?v=G8EBLFQZpwI
[download] Got error: HTTP Error 403: Forbidden
ERROR: fragment 1 not found, unable to continue
[download] Got error: HTTP Error 403: Forbidden
ERROR: fragment 1 not found, unable to continue
[download] Got error: HTTP Error 403: Forbidden
ERROR: fragment 1 not found, unable to continue
09:21:00 [INFO]    -> Đã lưu 28 comment.
09:21:00 [INFO] [2/16] Đang tải: https://www.youtube.com/watch?v=PoL3SwkUPTQ
09:21:04 [INFO]    -> Đã lưu 47 comment.
09:21:04 [INFO] [3/16] Đang tải: https://www.youtube.com/watch?v=-DamZEsJwBc
09:21:19 [INFO]    -> Đã lưu 564 comment.
09:21:19 [INFO] [4/16] Đang tải: https://www.youtube.com/watch?v=zPhVVYjw8aA
09:21:26 [INFO]    -> Đã lưu 190 comment.
09:21:26 [INFO] [5/16] Đang tải: https://www.youtube.com/watch?v=SrXrOuz0ggg
09:21:29 [INFO]    -> Đã lưu 7 comment.
09:21:29 [INFO] [6/16] Đang tải: https://www.youtube.com/watch?v=1eC1f2vF